# BDB27 — DL acceleration ceiling → in-game pass-rush pressure

**The runnable interface for the current task.** `TASK.md` is the binding spec. This notebook
opens, you set `MODE` (and, only if the data is not in the usual place, `DATA_DIR`) in the single
**Configuration** cell, then **Run All**. No terminal required.

**Provenance banner — read this before any number.**

* **`MODE = "full"`** — the final run. Every stage artefact is stamped `FULL`; only then may a
  number be quoted in the writeup. The headline block is live.
* **`MODE = "sample"`** — a plumbing run on the provided, truncated 2023 tracking sample
  (`2^20−1` rows). Every artefact is stamped `PLACEHOLDER_SAMPLE: UNVALIDATED SAMPLE OUTPUT`.
  **The headline block prints `WITHHELD`**: no sample number is printed, because a sample number
  must never be mistaken for a finding.
* **Missing full data is not a crash.** `game_tracking_{2023,24,25}.csv` do not exist on this
  host. The stages that need them are reported as `PENDING FULL RUN`, the rest of the run
  continues, and the notebook says exactly which files are absent.

The chain is exactly the DL chain in `TASK.md`:
`00_thresholds → 01_audit → 02_combine_ceiling → 03_game_snaps → 04_models → 05_validate`.
`05_validate` implements the **two items `TASK.md` Phase 5 names, and only those**: the
**within-player rest permutation** (`inter_snap_rest_s` shuffled inside each rusher-game, the
Part-1 mixed model refitted on the identical ladder, the observed load × rest interaction
compared with the permutation null) and the **context controls** (garbage time, the final two
minutes of each half, obvious passing situations). The retired spec's **ceiling permutation**
is kept in the code for provenance but is **not run**: every artefact labels it `SUPERSEDED`.
Any Phase-5 check that did not run is printed as `NOT RUN` / `NOT IMPLEMENTED` /`NOT ESTIMABLE`
(never as a pass), and one gap is reported rather than patched: this repository defines **no
rule for an "obvious passing situation"**, so that exclusion is `DEFINITION_UNDEFINED`. The
headline block prints Phase 5's status from `outputs/dl/validation/validation_result.json` and
nothing else; an absent or non-`FULL` artefact prints `NOT RUN`.


## 1. Configuration — edit only this cell

* `MODE="sample"` runs everything the provided data allows and marks the game-side stages
  `PENDING FULL RUN`.
* `MODE="full"` is the final analysis. It requires the three full tracking files and a frozen
  `notes/prereg.md`; if the tracking files are absent the dependent stages are skipped as
  `PENDING FULL RUN`, never run on the sample.
* `FORCE=True` rebuilds every stage. `FORCE_STAGES` rebuilds only the named stages.
  `RUN_ONLY` runs only the named stages and reports the rest as `SKIPPED — checkpoint`.
* `CLEAN=True` is destructive but **inert unless `CLEAN_CONFIRM="DELETE"`**. It deletes only
  `outputs/dl/` and `data/interim/dl_checkpoints/`; it never touches the retired LC chain's
  outputs.


In [ ]:
# ===================== CONFIG — edit only this cell =====================
MODE            = "sample"   # "sample" | "full"
DATA_DIR        = None       # None = auto-detect (/kaggle/input/... else <repo>/data/raw)
FORCE           = False      # True = force EVERY DL stage
FORCE_STAGES    = []         # e.g. ["03_game_snaps"]; [] = reuse checkpoints
RUN_ONLY        = []         # [] = every stage in DL_STAGE_ORDER; non-empty = only these stages
CLEAN           = False      # destructive; requires CLEAN_CONFIRM == "DELETE"
CLEAN_CONFIRM   = ""         # safety interlock
SHOW_FIGURES    = True       # render the optional diagnostic figures when the inputs exist
FIG_DPI         = 150

assert MODE in ("sample", "full"), MODE
assert DATA_DIR is None or isinstance(DATA_DIR, str), DATA_DIR
assert isinstance(FORCE, bool), FORCE
assert isinstance(FORCE_STAGES, (list, tuple)), FORCE_STAGES
assert all(isinstance(s, str) and s for s in FORCE_STAGES), FORCE_STAGES
assert isinstance(RUN_ONLY, (list, tuple)), RUN_ONLY
assert all(isinstance(s, str) and s for s in RUN_ONLY), RUN_ONLY
assert isinstance(CLEAN, bool), CLEAN
assert isinstance(CLEAN_CONFIRM, str), CLEAN_CONFIRM
assert isinstance(SHOW_FIGURES, bool), SHOW_FIGURES
assert isinstance(FIG_DPI, int) and FIG_DPI > 0, FIG_DPI

_EFFECTIVE_FORCE = ("ALL STAGES (FORCE=True)" if FORCE
                    else (list(FORCE_STAGES) or "none — reuse checkpoints"))
_CLEAN_ARMED = (CLEAN is True) and (CLEAN_CONFIRM == "DELETE")
print(f"mode              : {MODE}")
print(f"effective --force : {_EFFECTIVE_FORCE}")
print(f"RUN_ONLY          : {list(RUN_ONLY) or 'none — run every stage in DL_STAGE_ORDER'}")
print("CLEAN={} · CLEAN_CONFIRM={} — {}".format(
    CLEAN, "set" if CLEAN_CONFIRM else "empty",
    "ARMED: outputs/dl/ and the DL checkpoints will be deleted" if _CLEAN_ARMED
    else ("IGNORED: set CLEAN_CONFIRM='DELETE' to confirm" if CLEAN
          else "existing DL outputs are preserved")))
# ==========================================================================


## 2. Bootstrap — locate the repo, pick the data, write a scratch run-config

This cell finds the repository by walking upward until it sees `config.yaml` and
`src/dl/dl_common.py`, picks the data directory (Kaggle-aware), writes
`bdb27_run/dl_config.yaml` with **that absolute data path**, and prints a short environment block:
Python version, repo commit, data directory, and every required file present/absent.

A missing file is always named. Nothing is silently substituted.


In [ ]:
import copy
import importlib
import json
import shutil
import sys
from pathlib import Path

import pandas as pd
import yaml


def find_repo_root() -> Path:
    """Walk upward from the notebook's working directory."""
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "config.yaml").exists() and (parent / "src" / "dl" / "dl_common.py").exists():
            return parent
    raise RuntimeError(
        "Cannot locate the BDB27 repo root (need config.yaml + src/dl/dl_common.py). "
        "Open this notebook from inside the BDB27 checkout.")


REPO = find_repo_root()
SRC = REPO / "src"
LEGACY = SRC / "legacy_ceiling_v1"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
if str(LEGACY) not in sys.path:
    sys.path.append(str(LEGACY))          # stage_runner lives with the preserved chain

import dl.dl_common as D                   # noqa: E402
import stage_runner                        # noqa: E402  (D46: reload stages from disk)


def _git_sha() -> str:
    """Short HEAD sha, or an explicit 'unavailable' — never a silent blank."""
    try:
        import subprocess
        return subprocess.check_output(
            ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
            text=True, stderr=subprocess.DEVNULL).strip()
    except Exception as exc:               # git absent, not a repo, or permission denied
        return f"unavailable ({type(exc).__name__})"


CFG_REPO = D.load_config(str(REPO / "config.yaml"))
DL_CFG = D.dl(CFG_REPO)

BASE_FILES = ["combine_tracking.csv", "combine_results.csv", "players.csv",
              "games.csv", "player_play.csv"]
_sample_key = DL_CFG["mode"]["sample"]["tracking_key"]
_full_key = DL_CFG["mode"]["full"]["tracking_key"]
_sample_value = DL_CFG["files"][_sample_key]
_full_value = DL_CFG["files"][_full_key]
SAMPLE_TRACKING = (_sample_value if isinstance(_sample_value, str)
                   else str(_sample_value[0]))
FULL_TRACKING = (list(_full_value) if isinstance(_full_value, (list, tuple))
                 else [_full_value])


def _score(d: Path) -> int:
    return sum((d / f).exists() for f in BASE_FILES)


def detect_data_dir(override):
    """Explicit override, else Kaggle input, else <repo>/data/raw. Never guesses a path silently."""
    if override:
        p = Path(override).expanduser().resolve()
        if _score(p):
            return p
        hits = sorted(p.rglob("combine_tracking.csv")) if p.is_dir() else []
        return hits[0].parent if hits else p
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        hits = sorted(kaggle.rglob("combine_tracking.csv"))
        if hits:
            candidates = sorted({h.parent for h in hits},
                                key=lambda d: (_score(d), len(str(d))), reverse=True)
            return candidates[0]
    return REPO / "data" / "raw"


DATA = detect_data_dir(DATA_DIR)

# ---- scratch run-config: only the data path (and the active mode default) change ----
RUN_DIR = REPO / "bdb27_run"
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_CFG_PATH = RUN_DIR / "dl_config.yaml"
run_cfg = copy.deepcopy(CFG_REPO)
run_cfg.pop("_config_path", None)
run_cfg.setdefault("paths", {})["raw"] = str(DATA)
run_cfg["bdb27_dl"]["paths"]["raw"] = str(DATA)
run_cfg["bdb27_dl"]["mode"]["default"] = MODE
RUN_CFG_PATH.write_text(yaml.safe_dump(run_cfg, sort_keys=False), encoding="utf-8")
RUN_CFG = D.load_config(str(RUN_CFG_PATH))
DL_RUN = D.dl(RUN_CFG)


def _abs(path) -> Path:
    p = Path(path)
    return p if p.is_absolute() else (REPO / p)


OUTPUTS = _abs(DL_RUN["paths"]["outputs"])
FIGDIR = _abs(DL_RUN["paths"]["figures"])
CKPT = _abs(DL_RUN["paths"]["checkpoints"])
THRESH = _abs(DL_RUN["paths"]["thresholds"])
RUN_LOG = _abs(DL_RUN["paths"]["run_log"])
FIGDIR.mkdir(parents=True, exist_ok=True)

# ---- required-file inventory (list every absence) ---------------------------------
present = {f: (DATA / f).exists() for f in BASE_FILES}
SAMPLE_TRACKING_PRESENT = (DATA / SAMPLE_TRACKING).exists()
FULL_TRACKING_MISSING = [f for f in FULL_TRACKING if not (DATA / f).exists()]
FULL_TRACKING_READY = not FULL_TRACKING_MISSING
MISSING_BASE = [f for f in BASE_FILES if not present[f]]

# ---- CLEAN: only when explicitly armed; scoped to the DL chain --------------------
DL_CLEAN_DIRS = ("audit", "ceiling", "game", "models", "validation")
DL_CLEAN_FILES = ("thresholds.json",)
_CLEAN_ARMED = (CLEAN is True) and (CLEAN_CONFIRM == "DELETE")
if _CLEAN_ARMED:
    print("CLEAN=True confirmed — removing the DL chain outputs only:")
    for name in DL_CLEAN_DIRS:
        p = OUTPUTS / name
        n = sum(1 for f in p.rglob("*") if f.is_file()) if p.is_dir() else 0
        print(f"  {p.relative_to(REPO)}/  ({n} files)")
    for name in DL_CLEAN_FILES:
        p = OUTPUTS / name
        print(f"  {p.relative_to(REPO)}  ({1 if p.exists() else 0} files)")
    n_ckpt = len(list(CKPT.glob("*.json"))) if CKPT.is_dir() else 0
    print(f"  {CKPT.relative_to(REPO)}/  ({n_ckpt} DL checkpoints)")
    for name in DL_CLEAN_DIRS:
        shutil.rmtree(OUTPUTS / name, ignore_errors=True)
    for name in DL_CLEAN_FILES:
        try:
            (OUTPUTS / name).unlink()
        except FileNotFoundError:
            pass
    for f in OUTPUTS.glob("crash_*.txt"):
        f.unlink()
    if CKPT.is_dir():
        for f in CKPT.glob("*.json"):
            f.unlink()
elif CLEAN is True:
    print("CLEAN=True ignored: set CLEAN_CONFIRM='DELETE' to confirm. Outputs preserved.")


# ---- output readers used by the run and headline cells ----------------------------
def out_path(*parts) -> Path:
    return OUTPUTS.joinpath(*parts)


def read_csv(path, **kwargs) -> pd.DataFrame:
    path = Path(path)
    return pd.read_csv(path, **kwargs) if path.exists() else pd.DataFrame()


def read_json(path):
    """Parsed JSON, or None. A half-written file is 'absent', never a traceback."""
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return None


def prov_of(df: pd.DataFrame, fallback: str = "") -> str:
    if len(df) and "provenance" in df.columns:
        return str(df["provenance"].iloc[0])
    return fallback


def prov_of_json(path) -> str:
    return str((read_json(path) or {}).get("provenance", ""))


def to_num(series):
    return pd.to_numeric(series, errors="coerce")


print(f"python          : {sys.version.split()[0]} ({sys.platform})")
print(f"repo            : {REPO}")
print(f"repo sha        : {_git_sha()}")
print(f"data dir        : {DATA}")
print(f"mode            : {MODE}   (force={FORCE}, clean={CLEAN})")
print("required files  :")
for f in BASE_FILES:
    print(f"  [{'ok ' if present[f] else 'MISSING'}] {f}")
print(f"  [{'ok ' if SAMPLE_TRACKING_PRESENT else 'MISSING'}] {SAMPLE_TRACKING}   (sample tracking)")
for f in FULL_TRACKING:
    print(f"  [{'ok ' if (DATA / f).exists() else 'MISSING'}] {f}   (full tracking)")
if MISSING_BASE:
    print("  !! missing required base file(s): " + ", ".join(MISSING_BASE))
    print("  !! the Combine/player stages cannot run until these are supplied.")
if not SAMPLE_TRACKING_PRESENT:
    print(f"  !! sample tracking absent: {SAMPLE_TRACKING} — sample-mode game-side work is limited.")
print(f"run config      : {RUN_CFG_PATH.relative_to(REPO)}")
print(f"outputs         : {OUTPUTS.relative_to(REPO)}/")
print(f"figures         : {FIGDIR.relative_to(REPO)}/")
print(f"checkpoints     : {CKPT.relative_to(REPO)}/")
if MODE == "full":
    if FULL_TRACKING_READY:
        print("full-data gate  : all full tracking files present")
    else:
        print("full-data gate  : PENDING FULL RUN — absent: " + ", ".join(FULL_TRACKING_MISSING))
        print("                  stages 01_audit, 03_game_snaps and 04_models will be skipped")


## 3. Run the DL chain in `TASK.md` order

The order is exactly:

| # | stage | reads | writes |
|---|---|---|---|
| 1 | `00_thresholds` | config | `outputs/dl/thresholds.json` |
| 2 | `01_audit` | Combine + players/`player_play` (+ tracking where present) | `outputs/dl/audit/` |
| 3 | `02_combine_ceiling` | Combine tracking/results/players | `outputs/dl/ceiling/` |
| 4 | `03_game_snaps` | `player_play` + tracking | `outputs/dl/game/snaps/` |
| 5 | `04_models` | `03` snap table | `outputs/dl/models/` |
| 6 | `05_validate` | `04` + `03` outputs | `outputs/dl/validation/` |

Each stage is invoked through its **own `main()`**, in-process, with
`stage_runner.run_stage`, which purges the cached bytecode and reloads the stage and its helpers
from disk before every call. A `git pull` therefore cannot leave stale code in the kernel.

A stage that needs the absent full tracking files is reported as **`PENDING FULL RUN`** and is not
executed — it is not allowed to produce a `FULL`-stamped artefact from the sample. A genuine
exception is reported as `FAILED` with its traceback tail and crash-file path; the loop does not
abort.


In [ ]:
# ---- the DL chain; exactly the stages TASK.md specifies ------------------------------
DL_STAGE_ORDER = ["00_thresholds", "01_audit", "02_combine_ceiling",
                  "03_game_snaps", "04_models", "05_validate"]
STAGE_MODULES = {
    "00_thresholds": "dl.stage_00_thresholds",
    "01_audit": "dl.stage_01_audit",
    "02_combine_ceiling": "dl.stage_02_ceiling",
    "03_game_snaps": "dl.stage_03_snaps",
    "04_models": "dl.stage_04_models",
    "05_validate": "dl.stage_05_validate",
}
#: Stages that cannot be honest without the full tracking files.
FULL_DATA_STAGES = {"01_audit", "03_game_snaps", "04_models", "05_validate"}

DL_HELPERS = (
    "dl.dl_common", "dl.dl_ceiling_core", "dl.dl_load", "dl.dl_snaps",
    "dl.dl_blockers", "dl.dl_models_core", "dl.dl_survival",
    "dl.dl_audit_ab", "dl.dl_audit_ce", "dl.dl_audit_f", "dl.dl_audit_power",
)

# Crash reports land next to each stage's own outputs (outputs/dl/<sub>/crash_*.txt).
stage_runner.run_diagnostics.STAGE_OUTPUTS.update({
    "dl.stage_00_thresholds": ("", ("thresholds.json",)),
    "dl.stage_01_audit": ("audit", ("decision.json",)),
    "dl.stage_02_ceiling": ("ceiling", ("player_ceiling.csv", "summary.json")),
    "dl.stage_03_snaps": ("game/snaps", ("snaps.csv", "snaps_summary.json")),
    "dl.stage_04_models": ("models", ("model_result.json",)),
    "dl.stage_05_validate": ("validation", ("validation_result.json",)),
})


def _runlog_rows() -> int:
    if not RUN_LOG.exists():
        return 0
    try:
        return len(pd.read_csv(RUN_LOG))
    except Exception:
        return 0


def _peak_mb_after(offset: int, stage: str):
    """The stage's own peak RSS from outputs/run_log.csv (None when it was skipped)."""
    if not RUN_LOG.exists():
        return None
    try:
        frame = pd.read_csv(RUN_LOG)
    except Exception:
        return None
    new = frame.iloc[offset:]
    hit = new[(new["stage"] == stage) & (new["mode"] == MODE)]
    if not len(hit):
        return None
    values = pd.to_numeric(hit["peak_rss_mb"], errors="coerce").dropna()
    return float(values.iloc[-1]) if len(values) else None


def _run_dl_stage(name: str, extra_args):
    return stage_runner.run_stage(
        STAGE_MODULES[name], [*extra_args, "--config", str(RUN_CFG_PATH)],
        outputs_root=OUTPUTS, helpers=DL_HELPERS)


def _read(path):
    return read_json(path)


def stage_output_status(name: str) -> str:
    """Per-stage status read from the stage's own artefacts (never assumed)."""
    try:
        if name == "00_thresholds":
            obj = _read(THRESH)
            if obj is None:
                return "PENDING FULL RUN (no thresholds.json)"
            return f"OK · provenance={obj.get('provenance', '?')} · frozen={obj.get('frozen')}"
        if name == "01_audit":
            obj = _read(OUTPUTS / "audit" / "decision.json")
            if obj is None:
                return "PENDING FULL RUN (no audit/decision.json)"
            pending = ", ".join(obj.get("pending_items") or []) or "none"
            return f"{obj.get('gate', '?')} · pending items: {pending}"
        if name == "02_combine_ceiling":
            obj = _read(OUTPUTS / "ceiling" / "summary.json")
            if obj is None:
                return "PENDING FULL RUN (no ceiling/summary.json)"
            return (f"OK · {obj.get('n_players_with_fits', 0)} players · "
                    f"{obj.get('n_attempts_used', 0)} attempts · "
                    f"selected={obj.get('selected_features')}")
        if name == "03_game_snaps":
            obj = _read(OUTPUTS / "game" / "snaps" / "snaps_summary.json")
            if obj is None:
                return "PENDING FULL RUN (no game/snaps/snaps_summary.json)"
            return (f"OK · {obj.get('n_snaps', 0):,} snaps · "
                    f"event rate={obj.get('event_rate')} · reportable={obj.get('reportable')}")
        if name == "04_models":
            obj = _read(OUTPUTS / "models" / "model_result.json")
            if obj is None:
                return "PENDING FULL RUN (no models/model_result.json)"
            gate = (obj.get("gate") or {})
            return (f"{gate.get('verdict', '?')} · P1={gate.get('part1_estimable')} · "
                    f"P2={gate.get('part2_estimable')}")
        if name == "05_validate":
            obj = _read(OUTPUTS / "validation" / "validation_result.json")
            if obj is None:
                return "PENDING FULL RUN (no validation/validation_result.json)"
            perm = obj.get("ceiling_permutation") or {}
            missing = len(obj.get("not_implemented") or [])
            return (f"{obj.get('verdict', '?')} · ceiling_permutation="
                    f"{perm.get('status')} · not implemented={missing}")
    except Exception as exc:                # status printing must never kill the run
        return f"STATUS UNREADABLE ({type(exc).__name__}: {exc})"
    return "UNKNOWN STAGE"


_unknown_force = [s for s in FORCE_STAGES if s not in DL_STAGE_ORDER]
if _unknown_force:
    print(f"WARNING: FORCE_STAGES has unknown stage(s): {_unknown_force}")
RUN_ONLY_NORM = [str(s).strip() for s in (RUN_ONLY or []) if str(s).strip()]
_unknown_only = [s for s in RUN_ONLY_NORM if s not in DL_STAGE_ORDER]
if _unknown_only:
    print(f"WARNING: RUN_ONLY has unknown stage(s), ignored: {_unknown_only}")
STAGES_TO_RUN = [s for s in DL_STAGE_ORDER if (not RUN_ONLY_NORM or s in RUN_ONLY_NORM)]
SKIPPED_STAGES = [s for s in DL_STAGE_ORDER if s not in STAGES_TO_RUN]
if RUN_ONLY_NORM and not STAGES_TO_RUN:
    raise ValueError(f"RUN_ONLY={RUN_ONLY_NORM} matches no stage in DL_STAGE_ORDER")

if SKIPPED_STAGES:
    print(f"RUN_ONLY={RUN_ONLY_NORM} — running {len(STAGES_TO_RUN)}/{len(DL_STAGE_ORDER)} stages; "
          "the rest are SKIPPED and their checkpoints/outputs are reused:")
    for name in SKIPPED_STAGES:
        print(f"  [SKIPPED — checkpoint] {name}")
    print()

RUN_RESULTS = []
print(f"running DL chain in '{MODE}' mode — each stage reloads its code from disk ...\n")
for name in STAGES_TO_RUN:
    force = ["--force"] if (FORCE or name in FORCE_STAGES) else []

    # Never run a full-data stage on the truncated sample: it could stamp FULL
    # provenance from sample-sized input. Report the missing files instead.
    if MODE == "full" and name in FULL_DATA_STAGES and not FULL_TRACKING_READY:
        reason = "missing full tracking: " + ", ".join(FULL_TRACKING_MISSING)
        res = {"stage": name, "ok": False, "seconds": 0.0, "peak_mb": None,
               "status": "PENDING FULL RUN", "error": reason, "tail": [], "crash_file": None}
        RUN_RESULTS.append(res)
        print(f"[{'PENDING FULL RUN':<44}] {name:<18} {'—':>8}   {reason}")
        continue

    offset = _runlog_rows()
    try:
        res = _run_dl_stage(name, ["--mode", MODE, *force])
    except BaseException as exc:             # load-time failure: keep the loop alive
        import traceback
        res = {"stage": name, "ok": False, "seconds": 0.0, "peak_mb": None,
               "error": f"{type(exc).__name__}: {exc}", "tail": [],
               "traceback": traceback.format_exc(), "crash_file": None}
    res["peak_mb"] = _peak_mb_after(offset, name)

    if res.get("ok"):
        res["status"] = stage_output_status(name)
    else:
        error = str(res.get("error") or "")
        missing_input = ("FileNotFoundError" in error or "No such file" in error
                         or "not found" in error.lower())
        if MODE == "full" and name in FULL_DATA_STAGES and missing_input:
            res["status"] = "PENDING FULL RUN"
        else:
            res["status"] = "FAILED"
    RUN_RESULTS.append(res)

    ram = f"{res['peak_mb']:.1f} MB" if res.get("peak_mb") is not None else "—"
    print(f"[{res['status']:<44}] {name:<18} {res['seconds']:>7.2f}s  peak RAM {ram}")
    if not res.get("ok"):
        for line in (res.get("tail") or [])[-5:]:
            print(f"           {line}")
        if res.get("crash_file"):
            print(f"           crash report: {res['crash_file']}")

print()
print("per-stage result:")
for res in RUN_RESULTS:
    ram = f"{res['peak_mb']:.1f} MB" if res.get("peak_mb") is not None else "—"
    print(f"  {res['stage']:<18} {res['status']:<20} {res['seconds']:>7.2f}s  {ram}")
print("\n(Per-stage statuses come from the stages' own artefacts; a skipped or PENDING stage "
      "is not a failure.)")


## 4. Headline — the block to send back

Every value below is read from a file under `outputs/dl/`; none is typed here. A missing file or
column prints `NOT RUN — <field> absent`. A non-`FULL` provenance prints `NOT RUN` too: sample or
stale artefacts are never reported as a finding.

**In `MODE="sample"` the whole block prints `WITHHELD`.** That is deliberate: sample output is
`UNVALIDATED SAMPLE OUTPUT` and no sample number may be quoted or mistaken for a result.


In [ ]:
import math

print("HEADLINE RESULTS — DL acceleration ceiling → pass-rush pressure")
print("=" * 88)

if MODE != "full":
    print("WITHHELD — MODE='sample'.")
    print("Reason: every artefact produced in sample mode is stamped")
    print("        'PLACEHOLDER_SAMPLE: UNVALIDATED SAMPLE OUTPUT'.")
    print("        No sample number may be quoted as a result, so none is printed here.")
    print("        Set MODE='full' and re-run to obtain the pre-declared verdict.")
    # Estimability only — never an estimate. Whether a model *can* be fitted is a
    # property of the design and of the data's coverage, and it is the thing a
    # sample run is allowed to report (see notes/decisions.md D78).
    print("")
    print("ESTIMABILITY — structure only, no estimates (not a finding):")
    sample_models = read_json(OUTPUTS / "models" / "model_result.json")
    if sample_models is None:
        print("  NOT RUN — outputs/dl/models/model_result.json absent")
    else:
        sample_gate = sample_models.get("gate") or {}
        sample_derive = sample_models.get("derivation") or {}
        print(f"  gate verdict: {sample_gate.get('verdict')} (sample provenance)")
        print(f"  P1: {sample_models.get('part1_status')} · "
              f"rung {sample_models.get('part1_rung_used')}")
        print(f"  P2: {sample_models.get('part2_status')}")
        print(f"  derivation (§8 all-22 controls): {sample_derive.get('status')}")
    sample_validation = read_json(OUTPUTS / "validation" / "validation_result.json")
    if sample_validation is None:
        print("  Phase 5: NOT RUN — validation/validation_result.json absent")
    else:
        sample_perm = sample_validation.get("rest_permutation") or {}
        sample_ctx = sample_validation.get("context_controls") or {}
        print(f"  Phase 5: {sample_validation.get('verdict')} (sample provenance)")
        print(f"    within-player rest permutation: {sample_perm.get('status')} — "
              f"{sample_perm.get('reason') or 'no reason recorded'}")
        print(f"    context controls: {sample_ctx.get('status')}")
        for row in (sample_ctx.get("rows") or []):
            print(f"      [{row.get('status')}] {row.get('check')}: "
                  f"{row.get('reason') or ''}")
        sample_ceil = sample_validation.get("ceiling_permutation") or {}
        print(f"    ceiling permutation: {sample_ceil.get('status')} (retired spec)")
        not_impl = sample_validation.get("not_implemented") or []
        print(f"  Phase 5 checks NOT IMPLEMENTED: "
              f"{', '.join(not_impl) if not_impl else 'none'}")
    print("  every number stays withheld: sample output is UNVALIDATED.")
else:
    def _fmt(value, digits=4):
        if value is None:
            return "NA"
        try:
            f = float(value)
        except (TypeError, ValueError):
            return str(value)
        return f"{f:.{digits}f}" if math.isfinite(f) else "NA"

    def _short(value):
        if isinstance(value, dict):
            return ", ".join(f"{k}:{value[k]}" for k in list(value)[:8])
        if isinstance(value, (list, tuple)):
            return "[" + ", ".join(str(v) for v in value) + "]"
        return _fmt(value)

    def _artifact(path, label):
        """A JSON artefact that may be reported; None (with a NOT RUN line) otherwise."""
        obj = read_json(path)
        if obj is None:
            print(f"  NOT RUN — {label} absent")
            return None
        prov = str(obj.get("provenance", ""))
        if prov != "FULL":
            print(f"  NOT RUN — {label} provenance is {prov!r}, not 'FULL'")
            return None
        return obj

    def _csv_artifact(path, label):
        frame = read_csv(path)
        if not len(frame):
            print(f"  NOT RUN — {label} absent")
            return None
        prov = prov_of(frame)
        if prov != "FULL":
            print(f"  NOT RUN — {label} provenance is {prov!r}, not 'FULL'")
            return None
        return frame

    # ---- provenance gate -------------------------------------------------------
    thresholds = _artifact(THRESH, "outputs/dl/thresholds.json")
    if thresholds is None:
        print("\nHEADLINE WITHHELD — thresholds provenance is not FULL.")
        print("Nothing below would be reportable. Re-run MODE='full' with the full data.")
    else:
        audit = _artifact(OUTPUTS / "audit" / "decision.json",
                          "outputs/dl/audit/decision.json")
        ceiling = _artifact(OUTPUTS / "ceiling" / "summary.json",
                            "outputs/dl/ceiling/summary.json")
        models = _artifact(OUTPUTS / "models" / "model_result.json",
                           "outputs/dl/models/model_result.json")
        p1 = _artifact(OUTPUTS / "models" / "part1_result.json",
                       "outputs/dl/models/part1_result.json")
        p2 = _artifact(OUTPUTS / "models" / "part2_result.json",
                       "outputs/dl/models/part2_result.json")
        snaps_summary = _artifact(OUTPUTS / "game" / "snaps" / "snaps_summary.json",
                                  "outputs/dl/game/snaps/snaps_summary.json")

        print("\n1. AUDIT GO/NO-GO (Phase 1)")
        if audit is None:
            print("  NOT RUN — audit gate absent")
        else:
            print(f"  {audit.get('gate', 'NA')}")
            for reason in audit.get("reasons") or []:
                print(f"    - {reason}")
            counts = audit.get("counts") or {}
            if counts.get("n_matched") is None:
                print("  NOT RUN — counts.n_matched absent")
            else:
                total = counts.get("n_dl")
                print(f"  matched DL: {counts['n_matched']}"
                      + (f" / {total}" if total is not None else ""))
            if counts.get("a0_icc") is not None:
                print(f"  audit-reported a0 ICC: {_fmt(counts['a0_icc'], 4)}")

        print("\n2. COMBINE CEILING — a0")
        if ceiling is None:
            print("  NOT RUN — ceiling summary absent")
        else:
            icc = (ceiling.get("icc") or {}).get("a0")
            icc_se = (ceiling.get("icc_se") or {}).get("a0")
            if icc is None:
                print("  NOT RUN — icc.a0 absent")
            else:
                extra = f" (SE {_fmt(icc_se, 4)})" if icc_se is not None else ""
                print(f"  a0 ICC: {_fmt(icc, 4)}{extra}")
            gate_checks = ((models or {}).get("gate") or {}).get("checks") or {}
            icc_gate = gate_checks.get("a0_icc") or {}
            if icc_gate:
                print(f"  a0 ICC gate: [{icc_gate.get('status', 'NA')}] "
                      f"threshold {_short(icc_gate.get('threshold'))}")

        print("\n3. PRIMARY SNAP SET")
        gate_checks = ((models or {}).get("gate") or {}).get("checks") or {}
        if not gate_checks:
            print("  NOT RUN — gate.checks absent")
        else:
            for key in ("median_primary_snaps", "matched_dl"):
                item = gate_checks.get(key) or {}
                if item.get("value") is None:
                    print(f"  NOT RUN — gate.checks.{key}.value absent")
                else:
                    print(f"  {key}: {_short(item['value'])} "
                          f"[{item.get('status', 'NA')}] "
                          f"threshold {_short(item.get('threshold'))}")

        print("\n4. PART 1 — Combine a0 → time to pressure")
        if p1 is None:
            print("  NOT RUN — Part-1 result absent")
        else:
            estimate = p1.get("estimate") or {}
            if p1.get("status") == "OK" and estimate.get("estimate") is not None:
                print(f"  HR per 1 SD a0: {_fmt(estimate['estimate'], 4)} "
                      f"[{_fmt(estimate.get('lo'), 4)}, {_fmt(estimate.get('hi'), 4)}]")
                print(f"  rung: {p1.get('rung_used')} · n = {p1.get('n_obs')} snaps · "
                      f"events = {p1.get('n_events')} · players = {p1.get('n_players')}")
            else:
                print(f"  NOT RUN — Part-1 hazard ratio absent (status={p1.get('status')}; "
                      f"{p1.get('reason', 'no reason recorded')})")

        print("\n5. PART 2 — absolute vs relative load (out of sample)")
        if p2 is None:
            print("  NOT RUN — Part-2 result absent")
        elif p2.get("status") != "OK":
            print(f"  NOT RUN — Part-2 comparison absent (status={p2.get('status')}; "
                  f"{p2.get('reason', 'no reason recorded')})")
        else:
            absolute = (p2.get("absolute") or {}).get("mean_logscore")
            relative = (p2.get("relative") or {}).get("mean_logscore")
            print(f"  metric: mean held-out log score per snap (nats; higher is better)")
            print(f"  absolute: {_fmt(absolute, 5)} · relative: {_fmt(relative, 5)}")
            print(f"  difference (relative − absolute): {_fmt(p2.get('difference_nats_per_snap'), 5)} "
                  f"nats/snap, interval "
                  f"[{_fmt((p2.get('difference_interval') or [None])[0], 5)}, "
                  f"{_fmt((p2.get('difference_interval') or [None, None])[1], 5)}]")
            print(f"  verdict: {p2.get('verdict')} · better definition: "
                  f"{p2.get('better_definition')} · SOI {_fmt(p2.get('soi'), 5)} · "
                  f"equivalence margin {_fmt(p2.get('equivalence_margin'), 5)}")
            print(f"  scored snaps: {p2.get('n_scored_snaps')} · "
                  f"excludes zero: {p2.get('excludes_zero')} · exceeds SOI: {p2.get('exceeds_soi')}")

        print("\n6. S1 — population load coefficient and load × a0")
        s1 = (models or {}).get("s1") or {}
        if not s1:
            print("  NOT RUN — model_result.s1 absent")
        elif s1.get("status") != "OK":
            print(f"  NOT RUN — S1 absent (status={s1.get('status')}; "
                  f"{s1.get('reason', 'no reason recorded')})")
        else:
            naive = s1.get("naive") or {}
            corrected = s1.get("corrected") or {}
            print(f"  definition: {s1.get('definition')}")
            print(f"  load coefficient: {_fmt(naive.get('load_coef'), 5)} "
                  f"(SE {_fmt(naive.get('load_se'), 5)})")
            print(f"  load × a0 (naive): {_fmt(naive.get('interaction_coef'), 5)} "
                  f"(SE {_fmt(naive.get('interaction_se'), 5)})")
            if corrected.get("interaction_coef") is None:
                print(f"  load × a0 (corrected): NOT RUN — corrected.interaction_coef absent")
            else:
                print(f"  load × a0 (corrected): {_fmt(corrected['interaction_coef'], 5)} "
                      f"(SD across {corrected.get('n_draws')} draws "
                      f"{_fmt(corrected.get('interaction_sd'), 5)})")

        print("\n7. EVENT RATE AND MDE")
        event_rate = None
        event_source = None
        if p1 is not None and p1.get("event_rate") is not None:
            event_rate, event_source = p1["event_rate"], "Part-1 model frame"
        elif snaps_summary is not None and snaps_summary.get("event_rate") is not None:
            event_rate = snaps_summary["event_rate"]
            event_source = f"snap table (n={snaps_summary.get('n_snaps')})"
        if event_rate is None:
            print("  NOT RUN — event_rate absent")
        else:
            print(f"  event rate: {_fmt(event_rate, 4)} ({event_source})")
        mde = gate_checks.get("part1_mde") or {}
        if mde.get("value") is None:
            print("  NOT RUN — gate.checks.part1_mde.value absent")
        else:
            print(f"  Part-1 MDE: HR {_fmt(mde['value'], 4)} "
                  f"[{mde.get('status', 'NA')}] vs max MDE "
                  f"{_short(mde.get('threshold'))} · events {mde.get('events')}")
            if audit is not None:
                attenuated = (audit.get("counts") or {}).get("part1_mde_hr_attenuated")
                if attenuated is not None:
                    print(f"  attenuated MDE: HR {_fmt(attenuated, 4)}")

        print("\n8. PER-PLAYER RELIABILITY GATE")
        reliability = _csv_artifact(OUTPUTS / "models" / "reliability.csv",
                                    "outputs/dl/models/reliability.csv")
        if reliability is not None:
            for _, row in reliability.iterrows():
                estimate = row.get("estimate")
                if estimate is None or (isinstance(estimate, float) and not math.isfinite(estimate)):
                    print(f"  {row.get('quantity')}: NOT ESTIMABLE — estimate absent")
                else:
                    threshold = row.get("threshold")
                    threshold_text = ("threshold " + _short(threshold)
                                      if threshold is not None
                                      and not (isinstance(threshold, float) and math.isnan(threshold))
                                      else "no threshold")
                    print(f"  {row.get('quantity')}: {_fmt(estimate, 4)} ({threshold_text})")
        if s1.get("reliability") is not None if s1 else False:
            print(f"  S1 load-slope reliability: {_fmt(s1['reliability'], 4)}")

        print("\n9. PHASE 5 VALIDATION")
        validation = read_json(OUTPUTS / "validation" / "validation_result.json")
        if validation is None:
            print("  NOT RUN — outputs/dl/validation/validation_result.json absent")
        elif str(validation.get("provenance", "")) != "FULL":
            print(f"  NOT RUN — validation provenance is "
                  f"{validation.get('provenance')!r}, not 'FULL'")
        elif not (validation.get("checks") or {}):
            print("  NOT RUN — validation_result.json carries no checks block")
        else:
            print(f"  verdict: {validation.get('verdict')}")
            print(f"  {validation.get('verdict_statement')}")

            perm = validation.get("rest_permutation") or {}
            pdesign = perm.get("design") or {}
            pobs = perm.get("observed") or {}
            pinf = perm.get("inference") or {}
            if not perm:
                print("  NOT RUN — rest permutation absent from validation_result.json")
            elif perm.get("status") != "OK":
                print(f"  [{perm.get('status')}] within-player rest permutation: "
                      f"{perm.get('reason')}")
            else:
                print(f"  [OK] within-player rest permutation: observed "
                      f"{_fmt(pobs.get('coef_std'), 4)} · null mean "
                      f"{_fmt(pinf.get('null_mean'), 4)} · SD "
                      f"{_fmt(pinf.get('null_sd'), 4)} · "
                      f"p(two-sided) {_fmt(pinf.get('p_two_sided'), 4)} · "
                      f"p(one-sided) {_fmt(pinf.get('p_one_sided'), 4)} over "
                      f"{pinf.get('n_permutations')} refits "
                      f"(requested {pdesign.get('reps_requested')}, seed "
                      f"{pdesign.get('seed')})")
                print(f"       blocks: {pdesign.get('n_blocks')} · permuted "
                      f"{pdesign.get('n_blocks_permuted')} · unpermutable snaps "
                      f"{pdesign.get('n_snaps_in_singleton_blocks')} "
                      f"(counted, never dropped) · multiset preserved "
                      f"{pdesign.get('block_multiset_preserved')}")

            ctx = validation.get("context_controls") or {}
            if not (ctx.get("rows") or []):
                print("  NOT RUN — context controls absent from validation_result.json")
            else:
                print("  context controls (same ladder, restricted sample):")
                for row in ctx["rows"]:
                    est = row.get("estimate") or {}
                    print(f"    [{row.get('status')}] {row.get('check')}: "
                          f"interaction {_fmt(est.get('coef_std'), 4)} · "
                          f"n = {row.get('n_rows')} · excluded "
                          f"{row.get('n_excluded')}"
                          + (f" · {row.get('reason')}" if row.get('reason') else ""))

            ceil = validation.get("ceiling_permutation") or {}
            print(f"  [SUPERSEDED] ceiling permutation: {ceil.get('status')} — "
                  f"{ceil.get('reason') or ''}")
            for key, item in (validation.get("checks") or {}).items():
                if key in ("rest_permutation", "context_controls",
                           "ceiling_permutation"):
                    continue
                print(f"  [{item.get('status', 'NA')}] {key}: "
                      f"{item.get('reason') or ''}")
            missing = validation.get("not_implemented") or []
            print(f"  NOT IMPLEMENTED in this build ({len(missing)}): "
                  f"{', '.join(missing) if missing else 'none'}")

        print("\n10. GATE ITEMS — every pre-registered stopping condition")
        if not gate_checks:
            print("  NOT RUN — gate.checks absent")
        else:
            for key, item in gate_checks.items():
                value = item.get("value")
                value_text = "" if value is None else f" value={_short(value)}"
                threshold = item.get("threshold")
                threshold_text = ("" if threshold is None
                                  else f" threshold={_short(threshold)}")
                print(f"  [{item.get('status', 'NA')}] {key}{value_text}{threshold_text}")
                if item.get("clause"):
                    print(f"        clause: {item['clause']}")
                if item.get("reason"):
                    print(f"        reason: {item['reason']}")

        print("\n11. AUDIT ITEMS a–k")
        if audit is None:
            print("  NOT RUN — audit items absent")
        else:
            for key in sorted((audit.get("items") or {})):
                item = audit["items"][key]
                print(f"  [{item.get('status', 'NA')}] {key}: {item.get('reason', '')}")

print("\n" + "=" * 88)
print("End of headline. Missing items say NOT RUN; they are never filled with a guess or a zero.")


## 5. What to send back

If the run stops or something looks wrong, these are the artefacts that matter — send the files
themselves, not screenshots of the terminal:

1. **The headline cell output** (section 4 above). It is the plain-English verdict.
2. `outputs/dl/audit/decision.json` and `outputs/dl/audit/SUMMARY.md` — the go/no-go gate and
   every audit item with its evidence.
3. `outputs/dl/ceiling/summary.json` — the `a0` ICC, SE, the ICC/VIF screen and the selected
   feature set.
4. `outputs/dl/game/snaps/snaps_summary.json` — snap counts, exclusions, event rate, per-column
   `NA` rates and the all-22 derivation coverage.
5. `outputs/dl/models/model_result.json`, `part1_result.json`, `part2_result.json` and
   `reliability.csv` — the gate, P1, P2, S1 and the reliability rows.
6. `outputs/run_log.csv` (the DL rows) — wall time and peak RAM per stage.
7. If a stage is `FAILED`: the `crash_*.txt` path printed in section 3.
8. `notes/blockers.md` — the single place where open data/design gaps are recorded. In particular,
   `05_validate` is a known next build item and will print `NOT RUN` in section 4 until it exists.


## 6. Optional diagnostic figures (≤2)

Rendered only from `outputs/dl/` and only in `MODE="full"`. In sample mode they are skipped with a
reason: a sample plot is not a finding. If the needed columns are absent the figure is skipped
without an exception. These are diagnostics, not writeup figures.


In [ ]:
if not SHOW_FIGURES:
    print("figures: SHOW_FIGURES=False — skipped")
elif MODE != "full":
    print("figures: SKIPPED — MODE='sample'. A plot of sample output would not be a finding.")
else:
    try:
        import matplotlib
        import matplotlib.pyplot as plt
    except Exception as exc:
        print(f"figures: SKIPPED — matplotlib unavailable ({type(exc).__name__}: {exc})")
    else:
        made = 0

        # Figure 1 — absolute vs relative load (only when both have data).
        snaps = read_csv(OUTPUTS / "game" / "snaps" / "snaps.csv")
        if not len(snaps):
            print("figure 1: SKIPPED — outputs/dl/game/snaps/snaps.csv absent")
        elif prov_of(snaps) != "FULL":
            print(f"figure 1: SKIPPED — snaps provenance is {prov_of(snaps)!r}, not 'FULL'")
        else:
            absolute = to_num(snaps.get("absolute_load")) if "absolute_load" in snaps else pd.Series(dtype=float)
            relative = to_num(snaps.get("relative_load")) if "relative_load" in snaps else pd.Series(dtype=float)
            if absolute.notna().sum() == 0 or relative.notna().sum() == 0:
                print("figure 1: SKIPPED — absolute_load/relative_load absent or all NA")
            else:
                try:
                    fig, ax = plt.subplots(figsize=(8, 4.2))
                    ax.hist(absolute.dropna(), bins=30, alpha=0.55, label="absolute")
                    ax.hist(relative.dropna(), bins=30, alpha=0.55, label="relative")
                    ax.set_xlabel("cumulative load before snap")
                    ax.set_ylabel("snaps")
                    ax.set_title("Load definitions — FULL diagnostic")
                    ax.legend()
                    path = FIGDIR / "dl_fig1_load_comparison.png"
                    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
                    plt.show()
                    print(f"figure 1: saved -> {path}")
                    made += 1
                except Exception as exc:
                    print(f"figure 1: SKIPPED — {type(exc).__name__}: {exc}")

        # Figure 2 — Combine a0 vs in-game pressure rate (player level).
        ceiling = read_csv(OUTPUTS / "ceiling" / "player_ceiling.csv")
        if not len(ceiling) or not len(snaps):
            print("figure 2: SKIPPED — player_ceiling.csv or snaps.csv absent")
        elif prov_of(ceiling) != "FULL" or prov_of(snaps) != "FULL":
            print("figure 2: SKIPPED — ceiling/snaps provenance is not FULL for both")
        else:
            try:
                a0 = ceiling[ceiling["feature"] == "a0"][["nfl_id", "estimate"]].copy()
                events = (snaps.assign(event=to_num(snaps["event"]))
                          .groupby("nfl_id", as_index=False)["event"].mean()
                          .rename(columns={"event": "pressure_rate"}))
                merged = a0.merge(events, on="nfl_id", how="inner")
                merged = merged[pd.to_numeric(merged["estimate"], errors="coerce").notna()]
                if len(merged) < 5:
                    print(f"figure 2: SKIPPED — only {len(merged)} players with both a0 and snaps")
                else:
                    fig, ax = plt.subplots(figsize=(7.5, 4.4))
                    ax.scatter(to_num(merged["estimate"]), to_num(merged["pressure_rate"]),
                               s=18, alpha=0.7)
                    ax.set_xlabel("shrunken a0 (yd/s²)")
                    ax.set_ylabel("pressure rate per snap")
                    ax.set_title("Combine ceiling vs in-game pressure — FULL diagnostic")
                    path = FIGDIR / "dl_fig2_ceiling_vs_pressure.png"
                    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
                    plt.show()
                    print(f"figure 2: saved -> {path}")
                    made += 1
            except Exception as exc:
                print(f"figure 2: SKIPPED — {type(exc).__name__}: {exc}")

        print(f"figures made: {made} (diagnostics only, not writeup figures)")
